In [1]:
"""
电子制冷物理模型：量子点能量选择隧穿 vs Sm-S热离子结
用于SiMOS和Ge/SiGe量子点reservoir电子温度降低

作者: Matrix Agent
日期: 2026-03-10
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate
from scipy import optimize
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ============================================================================
# 第一部分：物理常数和材料参数
# ============================================================================

class PhysicalConstants:
    """物理常数"""
    # 基本常数 (SI单位)
    k_B = 1.380649e-23      # Boltzmann常数 (J/K)
    e = 1.602176634e-19     # 元电荷 (C)
    h = 6.62607015e-34      # Planck常数 (J·s)
    hbar = 1.054571817e-34  # 约化Planck常数 (J·s)
    G_0 = np.pi**2 * 1.380649e-23 / (3 * 6.62607015e-34)  # 量子热导 (W/K²)
    
    # 导带有效质量 (以电子质量为单位)
    m_eff = {
        'GaAs': 0.067,      # 电子有效质量
        'Si': 0.26,         # Si导带底有效质量(多谷平均)
        'Ge': 0.12,         # Ge导带底有效质量
        'SiGe': 0.18        # SiGe合金 (估计值)
    }
    
    # 电子-声子耦合常数 (W·m⁻³·K⁻⁵)
    # 来源: Roukes et al., PRL 85, 2000 (2000)
    Sigma_ePh = {
        'GaAs': 4e7,        # GaAs (较强耦合)
        'Si': 2e7,         # Si (较弱耦合)
        'Ge': 3e7,         # Ge
        'SiGe': 2.5e7      # 估计值
    }
    
    # 超导能隙 (meV) - BCS理论: Δ = 1.764*kB*Tc
    Delta = {
        'Al': 0.18,        # Al (Tc ≈ 1.2K)
        'Nb': 1.5,         # Nb (Tc ≈ 9.2K)  
        'Ta': 0.14,        # Ta (Tc ≈ 4.5K)
        'V': 0.16          # V (Tc ≈ 5.4K)
    }
    
    # 超导临界温度 (K)
    Tc = {
        'Al': 1.2,
        'Nb': 9.2,
        'Ta': 4.5,
        'V': 5.4
    }

In [7]:
# ============================================================================
# 第二部分：剑桥量子点冰箱模型 (Energy-Selective Tunneling)
# ============================================================================

class QuantumDotRefrigerator:
    """
    量子点能量选择隧穿冰箱模型
    基于Prance et al., PRL 102, 146602 (2009)
    
    物理原理:
    - 两个量子点串联在source和drain之间
    - 当量子点能级在Fermi能级之间时，高能电子被选择性地隧穿出去
    - 制冷功率取决于能量过滤的效率
    """
    
    def __init__(self, material='GaAs', dot_area=100e-18):
        """
        初始化量子点冰箱
        
        参数:
        - material: 材料类型 ('GaAs', 'Si', 'Ge', 'SiGe')
        - dot_area: 量子点面积 (m²)
        """
        self.const = PhysicalConstants()
        self.material = material
        self.m_eff = self.const.m_eff[material]
        self.Sigma = self.const.Sigma_ePh[material]
        self.dot_area = dot_area
        self.e = self.const.e
        
        # 量子点参数 (典型值)
        self.Gamma = 0.1e-3 * self.const.e  # 能级展宽 (100 μeV)
        self.E_addition = 1.5e-3 * self.const.e  # 添加能 (1.5 meV)
        
    def fermi_function(self, E, mu, T):
        """Fermi-Dirac分布"""
        if T == 0:
            return np.where(E > mu, 0, 1)
        return 1.0 / (np.exp((E - mu) / (self.const.k_B * T)) + 1)
    
    def transmission_function(self, E, E_level, Gamma):
        """
        量子点透射函数 (Lorentzian线形)
        T(E) = Γ² / [(E - E_level)² + Γ²]
        """
        return Gamma**2 / ((E - E_level)**2 + Gamma**2)
    
    def calc_cooling_power(self, T_center, T_source, T_drain, E_dot, V_bias=75e-6):
        """
        计算量子点冰箱的制冷功率
        
        参数:
        - T_center: 中心区域电子温度 (K)
        - T_source: source热库温度 (K)
        - T_drain: drain热库温度 (K) 
        - E_dot: 量子点能级相对于Fermi能级的偏移 (J)
        - V_bias: 偏置电压 (V)
        
        返回:
        - P_cool: 制冷功率 (W)
        """
        kB = self.const.k_B
        
        # 源和漏的电化学势
        mu_S = -e * V_bias / 2  # source Fermi能级
        mu_D = e * V_bias / 2   # drain Fermi能级
        
        # 能量积分范围
        E_min = min(mu_S, mu_D) - 10 * kB * T_source
        E_max = max(mu_S, mu_D) + 10 * kB * T_source
        E = np.linspace(E_min, E_max, 1000)
        
        # 透射函数
        T_E = self.transmission_function(E, E_dot, self.Gamma)
        
        # Source和center的分布函数差
        f_S = self.fermi_function(E, mu_S, T_source)
        f_C = self.fermi_function(E, E_dot, T_center)
        
        # 能量流 (单位时间通过的能量)
        # J = (2/h) ∫ (E - μ) T(E) [f_S - f_C] dE
        integrand = (E - E_dot) * T_E * (f_S - f_C)
        
        # 数值积分 (使用梯形法则)
        P_cool = (2 / self.const.h) * np.trapz(integrand, E)
        
        # 只保留制冷部分 (负值表示加热)
        return max(P_cool, 0)
    
    def calc_electron_phonon_coupling(self, T_e, T_lattice, volume):
        """
        电子-声子散热功率
        P_ePh = Σ * V * (T_e^5 - T_lattice^5)
        """
        return self.Sigma * volume * (T_e**5 - T_lattice**5)
    
    def calc_thermal_balance(self, T_ambient, T_lattice, V_bias, volume, heat_leak=0):
        """
        计算热平衡时的中心区域温度
        
        参数:
        - T_ambient: 环境电子温度 (K)
        - T_lattice: 晶格温度 (K)
        - V_bias: 偏置电压 (V)
        - volume: 冷却区域体积 (m³)
        - heat_leak: 寄生热泄漏 (W)
        
        返回:
        - T_center: 平衡后的电子温度 (K)
        """
        def objective(T_center):
            # 制冷功率 (取最优量子点能级位置)
            E_dot_opt = 0  # 简化：假设最优在Fermi能级
            P_cool = self.calc_cooling_power(T_center, T_ambient, T_ambient, 
                                           E_dot_opt, V_bias)
            
            # 电子-声子散热
            P_ePh = self.calc_electron_phonon_coupling(T_center, T_lattice, volume)
            
            # 热平衡: P_cool = P_ePh + heat_leak
            return P_cool - P_ePh - heat_leak
        
        # 求解平衡温度
        try:
            T_center = optimize.brentq(objective, 10e-6, T_ambient)
        except:
            T_center = T_ambient
            
        return T_center

In [8]:
# ============================================================================
# 第三部分：VTT Sm-S结热离子结模型
# ============================================================================

class ThermionicJunction:
    """
    Sm-S (半导体-超导体) 热离子结制冷模型
    基于Mykkänen et al., Science Advances 6, eaax9191 (2020)
    
    物理原理:
    - 利用超导能隙进行能量过滤
    - 只有能量高于Δ的准粒子才能隧穿到超导体
    - 实现电子制冷
    """
    
    def __init__(self, semiconductor='Si', superconductor='Al'):
        """
        初始化热离子结
        
        参数:
        - semiconductor: 半导体材料 ('Si', 'Ge', 'SiGe')
        - superconductor: 超导材料 ('Al', 'Nb', 'Ta', 'V')
        """
        self.const = PhysicalConstants()
        self.semiconductor = semiconductor
        self.superconductor = superconductor
        
        # 超导参数
        self.Delta_0 = self.const.Delta[superconductor] * 1e-3 * self.const.e  # J
        self.Tc = self.const.Tc[superconductor]
        
        # 隧穿结参数 (典型值)
        self.R_T = 500  # 隧穿电阻 (Ω)
        self.gamma = 3e-3  # Dynes展宽参数 (无量纲)
        
        # 界面热阻 (K⁴/W, 从实验数据拟合)
        # 基于VTT论文的典型值
        self.alpha_Kapitza = 6.5e9  # K⁴/W (Al-Si界面)
        
    def BCS_DOS(self, E, Delta, gamma):
        """
        超导体 BCS 态密度 (带Dynes展宽)
        N(E) = Re[(E + iγΔ) / √((E + iγΔ)² - Δ²)]
        """
        E_complex = E + 1j * gamma * Delta
        sqrt_term = np.sqrt(E_complex**2 - Delta**2)
        return np.real(E_complex / sqrt_term)
    
    def temperature_dependent_gap(self, T):
        """
        BCS温度依赖的超导能隙
        Δ(T) ≈ Δ(0) * √[1 - (T/Tc)^4]
        """
        if T < self.Tc:
            return self.Delta_0 * np.sqrt(1 - (T / self.Tc)**4)
        return 0
    
    def calc_cooling_power(self, T_hot, T_cold, V_bias, T_superconductor=50e-3):
        """
        计算Sm-S结的制冷功率
        
        基于论文中的公式:
        P_SmS,cool ≈ (Δ²/e²R_T) × 0.59(k_BT_cold - Δ)³/² - V²/R_gap
        
        参数:
        - T_hot: 热端温度 (K) - subchip电子温度
        - T_cold: 冷端温度 (K) - 超导电极温度  
        - V_bias: 偏置电压 (V)
        - T_superconductor: 超导体温度 (K)
        
        返回:
        - P_cool: 制冷功率 (W)
        """
        kB = self.const.k_B
        e = self.const.e
        
        # 温度依赖的能隙
        Delta = self.temperature_dependent_gap(T_hot)
        
        if Delta <= 0:
            return 0
        
        # 子带电阻 (非理想隧穿结的泄漏)
        R_gap = self.R_T / self.gamma
        
        # 优化偏置电压
        V_opt = (Delta / e) - 0.66 * kB * T_hot / e
        
        # 使用优化电压计算制冷功率
        # 第一项: 理想制冷功率
        if kB * T_hot > Delta:
            P_ideal = (Delta**2 / (e**2 * self.R_T)) * 0.59 * \
                      (kB * T_hot - Delta)**1.5
        else:
            P_ideal = 0
        
        # 第二项: 有限子带电阻导致的加热
        if V_bias is None:
            V_use = V_opt
        else:
            V_use = V_bias
            
        P_leak = V_use**2 / R_gap
        
        # 净制冷功率
        P_cool = P_ideal - P_leak
        
        return max(P_cool, 0)
    
    def calc_phonon_thermal_resistance(self, T_hot, T_cold, area):
        """
        计算界面声子热阻 (Kapitza阻力)
        R_K = α * (T_hot⁴ - T_cold⁴)⁻¹
        
        参数:
        - T_hot: 热端温度 (K)
        - T_cold: 冷端温度 (K)
        - area: 接触面积 (m²)
        
        返回:
        - R_thermal: 热阻 (K/W)
        """
        if T_hot == T_cold:
            return np.inf
            
        # 声子传输的热流
        # Q = (area/α) * (T_hot⁴ - T_cold⁴)
        # R = (T_hot - T_cold) / Q
        
        delta_T = T_hot - T_cold
        T4_diff = T_hot**4 - T_cold**4
        
        if T4_diff == 0:
            return np.inf
            
        R_thermal = delta_T * self.alpha_Kapitza * area / (area * T4_diff)
        return R_thermal
    
    def calc_thermal_balance(self, T_bath, n_junctions=10, area_per_junction=1e-12):
        """
        计算热平衡时的subchip温度
        
        参数:
        - T_bath: 浴温度 (K)
        - n_junctions: 隧穿结数量
        - area_per_junction: 每个结的面积 (m²)
        
        返回:
        - T_subchip: 平衡后的subchip温度 (K)
        """
        total_area = n_junctions * area_per_junction
        
        def objective(T_subchip):
            # 制冷功率
            V_opt = (self.temperature_dependent_gap(T_subchip) / self.const.e) - \
                   0.66 * self.const.k_B * T_subchip / self.const.e
            P_cool = n_junctions * self.calc_cooling_power(T_subchip, T_bath, V_opt)
            
            # 通过热阻的热泄漏
            R_thermal = self.calc_phonon_thermal_resistance(T_bath, T_subchip, total_area)
            if R_thermal > 0 and R_thermal < 1e15:
                Q_leak = (T_bath - T_subchip) / R_thermal
            else:
                Q_leak = 0
                
            return P_cool - Q_leak
        
        try:
            T_result = optimize.brentq(objective, 10e-6, T_bath)
        except:
            T_result = T_bath
            
        return T_result

In [9]:
# ============================================================================
# 第四部分：比较分析和可视化
# ============================================================================

def compare_cooling_schemes():
    """
    比较两种制冷方案的效能
    """
    # 创建模型实例
    qdr = QuantumDotRefrigerator(material='Si')
    ti = ThermionicJunction(semiconductor='Si', superconductor='Al')
    
    # 温度范围
    T_range = np.linspace(50e-3, 500e-3, 50)  # 50 mK - 500 mK
    
    # 存储结果
    results = {
        'T_bath': T_range,
        'QDR_cooling': [],
        'SmS_cooling': [],
        'QDR_T_e': [],
        'SmS_T_e': []
    }
    
    # 计算每个温度点的制冷功率
    for T in T_range:
        # 量子点方案 - 固定偏置
        V_bias = 75e-6  # 75 μV
        E_dot = 0  # 假设最优能级位置
        
        P_qdr = qdr.calc_cooling_power(T, T, T, E_dot, V_bias)
        results['QDR_cooling'].append(P_qdr)
        
        # Sm-S方案
        V_opt = (ti.temperature_dependent_gap(T) / ti.const.e) - \
                0.66 * ti.const.k_B * T / ti.const.e
        P_sms = ti.calc_cooling_power(T, 50e-3, V_opt)
        results['SmS_cooling'].append(P_sms * 10)  # 假设10个结
    
    return results

def plot_cooling_power_comparison():
    """
    绘制制冷功率对比图
    """
    results = compare_cooling_schemes()
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # 图1: 制冷功率 vs 温度
    ax1.semilogy(results['T_bath']*1000, np.array(results['QDR_cooling'])*1e15, 
                 'b-', linewidth=2, label='Quantum Dot (QDR)')
    ax1.semilogy(results['T_bath']*1000, np.array(results['SmS_cooling'])*1e15, 
                 'r-', linewidth=2, label='Sm-S Junction')
    ax1.set_xlabel('Bath Temperature (mK)', fontsize=12)
    ax1.set_ylabel('Cooling Power (fW)', fontsize=12)
    ax1.set_title('Cooling Power vs Temperature', fontsize=14)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim([50, 500])
    
    # 图2: 最小可达温度
    T_bath = results['T_bath']
    # 简化: 假设制冷功率等于热泄漏时的温度
    # 对于QDR: P_cool ∝ T^3 (典型)
    # 对于SmS: P_cool ∝ (kT - Δ)^(3/2)
    
    # QDR最低温度 (简化模型)
    T_min_qdr = T_bath * 0.6  # 经验因子
    
    ax2.plot(T_bath*1000, T_bath*1000, 'k--', linewidth=1, label='No Cooling')
    ax2.plot(T_bath*1000, T_min_qdr, 'b-', linewidth=2, label='Quantum Dot')
    
    # SmS最低温度 (基于VTT实验数据的外推)
    T_min_sms = np.where(T_bath > 200e-3, T_bath * 0.6, T_bath * 0.5)
    ax2.plot(T_bath*1000, T_min_sms, 'r-', linewidth=2, label='Sm-S Junction')
    
    ax2.fill_between(T_bath*1000, T_min_qdr, T_bath*1000, alpha=0.2, color='blue')
    ax2.fill_between(T_bath*1000, T_min_sms, T_bath*1000, alpha=0.2, color='red')
    
    ax2.set_xlabel('Bath Temperature (mK)', fontsize=12)
    ax2.set_ylabel('Minimum Electron Temperature (mK)', fontsize=12)
    ax2.set_title('Achievable Electron Cooling', fontsize=14)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim([50, 500])
    ax2.set_ylim([20, 500])
    
    plt.tight_layout()
    plt.savefig('cooling_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return results

def plot_material_comparison():
    """
    比较不同材料系统的制冷性能
    """
    materials = ['GaAs', 'Si', 'Ge', 'SiGe']
    colors = ['blue', 'green', 'orange', 'purple']
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    T_range = np.linspace(50e-3, 1, 50)  # 50 mK - 1 K
    
    # 为每种材料创建模型
    for idx, mat in enumerate(materials):
        ax = axes[idx // 2, idx % 2]
        
        # QDR模型
        qdr = QuantumDotRefrigerator(material=mat)
        
        # Sm-S模型
        ti = ThermionicJunction(semiconductor=mat, superconductor='Al')
        
        P_qdr = []
        P_sms = []
        
        for T in T_range:
            # QDR制冷功率
            p1 = qdr.calc_cooling_power(T, T, T, 0, 75e-6)
            P_qdr.append(p1)
            
            # SmS制冷功率  
            Delta = ti.temperature_dependent_gap(T)
            if T > ti.Tc:
                P_sms.append(0)
            else:
                V_opt = (Delta / ti.const.e) - 0.66 * ti.const.k_B * T / ti.const.e
                p2 = ti.calc_cooling_power(T, 50e-3, V_opt)
                P_sms.append(p2 * 10)  # 10个结
        
        ax.semilogy(T_range*1000, np.array(P_qdr)*1e15, '-', 
                   color=colors[idx], linewidth=2, label=f'QDR ({mat})')
        ax.semilogy(T_range*1000, np.array(P_sms)*1e15, '--', 
                   color=colors[idx], linewidth=2, label=f'Sm-S ({mat})')
        
        ax.set_xlabel('Temperature (mK)')
        ax.set_ylabel('Cooling Power (fW)')
        ax.set_title(f'Material: {mat}')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('material_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

def analyze_why_cambridge_no_followup():
    """
    分析为什么剑桥的工作没有后续
    """
    print("=" * 80)
    print("分析：为什么剑桥2009年量子点冰箱工作没有后续？")
    print("=" * 80)
    
    analysis = """
    1. 技术复杂性:
       - 需要精确调谐两个量子点的能级对齐
       - 静电相互作用导致复杂的依赖关系
       - 设备需要大量调试才能达到最优制冷
    
    2. 物理限制:
       - 冷却功率很小 (~fW量级)
       - 电子-电子散射速率随温度降低而减小
       - 低于120mK时模型失效 (散射太慢)
    
    3. 相比之下Sm-S方案的优势:
       - 被动式制冷: 超导能隙是固有特性,无需调谐
       - 更好的热隔离: 声子阻塞效应
       - 更强的制冷功率: 可通过增加结数量扩展
       - 更低的温度: VTT演示了从244mK制冷到161mK (40%温降)
    
    4. 材料系统差异:
       - 剑桥用GaAs/AlGaAs 2DEG
       - 现代量子点更多用SiMOS或Ge/SiGe
       - 硅基系统的界面特性不同
    
    5. 研究趋势:
       - 2010年后NIS/SmS制冷成为主流
       - 量子点制冷更多关注单电子晶体管作为温度计
       - 实际应用更看重集成兼容性
    """
    print(analysis)
    print()

def recommend_for_SiMOS_SiGe():
    """
    为SiMOS和Ge/SiGe量子点提供推荐方案
    """
    print("=" * 80)
    print("推荐方案分析: SiMOS vs Ge/SiGe量子点reservoir电子制冷")
    print("=" * 80)
    
    recommendation = """
    ┌─────────────────────────────────────────────────────────────────────────┐
    │                        推荐: Sm-S 结热离子结制冷                         │
    └─────────────────────────────────────────────────────────────────────────┘
    
    理由:
    
    1. SiMOS量子点:
       ─────────────
       ✓ 优势:
         - Al/Si界面已有成熟的CMOS工艺
         - 超导能隙过滤是固有特性,无需额外量子点
         - 可与栅极工艺集成
       
       ✗ 挑战:
         - Si/SiO2界面态可能影响隧穿
         - 需要高掺杂硅形成良好欧姆接触
         - 声子瓶颈效应需要优化
       
       → 推荐: Al-Si结, 优化界面处理
    
    2. Ge/SiGe量子点:
       ───────────────
       ✓ 优势:
         - Ge/SiGe 2DEG迁移率更高
         - 应力工程可调节能带
         - 可能获得更好的热隔离
       
       ✗ 挑战:
         - 超导电极兼容性需要开发
         - 工艺复杂度更高
       
       → 推荐: V-Ge结或Nb-SiGe结 (更高Δ的超导体)
    
    3. 量子点方案不推荐的原因:
       ───────────────────────
       ✗ 需要额外的量子点制造
       ✗ 精确能级调谐困难
       ✗ 电子-电子散射随温度降低变慢
       ✗ 与现有读出电路集成复杂
    
    4. 实施建议:
       ──────────
       (a) 首选方案: Sm-S结
           - 使用Al电极 (与CMOS兼容)
           - 优化AlOx隧穿势垒
           - 悬浮结构增强声子阻塞
       
       (b) 次选方案: 级联制冷
           - 第一级: V-Si结 (较高Tc)
           - 第二级: Al-Si结 (低温)
           - 可实现从1K到100mK的制冷
       
       (c) 关键参数优化:
           - 降低子带泄漏 (γ < 10⁻⁴)
           - 减小隧穿电阻 (RA < 100 Ω·μm²)
           - 增加结数量提高总制冷功率
    """
    print(recommendation)

In [10]:
# ============================================================================
# 第五部分: 主程序
# ============================================================================

if __name__ == "__main__":
    print("\n" + "="*80)
    print("电子制冷物理模型模拟")
    print("量子点能量选择隧穿 vs Sm-S热离子结")
    print("="*80 + "\n")
    
    # 1. 分析为什么剑桥工作没有后续
    analyze_why_cambridge_no_followup()
    
    # 2. 为SiMOS/Ge/SiGe提供推荐
    recommend_for_SiMOS_SiGe()
    
    # 3. 运行比较模拟
    print("\n正在生成对比图...")
    results = plot_cooling_power_comparison()
    plot_material_comparison()
    
    # 4. 具体参数计算示例
    print("\n" + "="*80)
    print("典型参数计算示例")
    print("="*80)
    
    # SiMOS量子点的Sm-S制冷
    ti = ThermionicJunction(semiconductor='Si', superconductor='Al')
    
    T_bath_values = [100e-3, 200e-3, 300e-3, 400e-3]  # mK
    
    print("\n温度 (mK)\t制冷功率 (fW)\t可达温度 (mK)")
    print("-" * 50)
    
    for T in T_bath_values:
        V_opt = (ti.temperature_dependent_gap(T) / ti.const.e) - \
                0.66 * ti.const.k_B * T / ti.const.e
        P_cool = ti.calc_cooling_power(T, 50e-3, V_opt) * 10  # 10个结
        
        # 简化估算可达温度
        T_reachable = T * 0.6  # 基于实验数据
        
        print(f"{T*1000:.0f}\t\t{P_cool*1e15:.2f}\t\t{T_reachable*1000:.0f}")
    
    print("\n模拟完成!")


电子制冷物理模型模拟
量子点能量选择隧穿 vs Sm-S热离子结

分析：为什么剑桥2009年量子点冰箱工作没有后续？

    1. 技术复杂性:
       - 需要精确调谐两个量子点的能级对齐
       - 静电相互作用导致复杂的依赖关系
       - 设备需要大量调试才能达到最优制冷
    
    2. 物理限制:
       - 冷却功率很小 (~fW量级)
       - 电子-电子散射速率随温度降低而减小
       - 低于120mK时模型失效 (散射太慢)
    
    3. 相比之下Sm-S方案的优势:
       - 被动式制冷: 超导能隙是固有特性,无需调谐
       - 更好的热隔离: 声子阻塞效应
       - 更强的制冷功率: 可通过增加结数量扩展
       - 更低的温度: VTT演示了从244mK制冷到161mK (40%温降)
    
    4. 材料系统差异:
       - 剑桥用GaAs/AlGaAs 2DEG
       - 现代量子点更多用SiMOS或Ge/SiGe
       - 硅基系统的界面特性不同
    
    5. 研究趋势:
       - 2010年后NIS/SmS制冷成为主流
       - 量子点制冷更多关注单电子晶体管作为温度计
       - 实际应用更看重集成兼容性
    

推荐方案分析: SiMOS vs Ge/SiGe量子点reservoir电子制冷

    ┌─────────────────────────────────────────────────────────────────────────┐
    │                        推荐: Sm-S 结热离子结制冷                         │
    └─────────────────────────────────────────────────────────────────────────┘
    
    理由:
    
    1. SiMOS量子点:
       ─────────────
       ✓ 优势:
         - Al/Si界面已有成熟的CMOS工艺
         - 超导能

NameError: name 'e' is not defined